# 🎓 Заняття 5. Побудова RAG-систем

In [1]:
!pip install "langchain>=1.2.12" "langchain-core>=1.2.19" "langchain-openai>=0.4" "langchain-community>=0.4" "langchain-experimental" "langgraph>=0.4" "llama-index>=0.14.16" "llama-index-core>=0.14.16" "llama-index-llms-openai>=0.6.23" "llama-index-embeddings-openai>=0.4" "llama-index-vector-stores-qdrant>=0.6" "faiss-cpu" "rank_bm25" "sentence-transformers" "duckduckgo-search" "pypdf" "matplotlib"

In [2]:
import os
import logging
import warnings
from getpass import getpass

import nest_asyncio
nest_asyncio.apply()

# Suppress noisy logs
for logger_name in ["asyncio", "httpx", "openai", "huggingface_hub", "sentence_transformers"]:
    logging.getLogger(logger_name).setLevel(logging.CRITICAL)
warnings.filterwarnings("ignore", message="coroutine.*was never awaited")
warnings.filterwarnings("ignore", message=".*unauthenticated requests.*")

# Suppress HuggingFace model load reports
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

print("✅ API key configured!")

Enter your OpenAI API key:  ········


✅ API key configured!


---
# 📚 Еволюція RAG та Інструментарій

## Проблема контекстного вікна (Context Window Problem)

### Чому не можна просто завантажити все в промпт?

**3 головні проблеми:**

| Проблема | Пояснення |
|----------|-----------|
| 💰 **Вартість** | 1M input tokens для GPT-5.4 = $2.50. 500 сторінок документації при кожному запиті — фінансова катастрофа |
| 🐌 **Швидкість** | Більше токенів = повільніша генерація (latency росте лінійно з розміром контексту) |
| 🧠 **Складність reasoning** | При дуже довгому контексті (500K+ токенів) моделі можуть втрачати точність у multi-hop reasoning задачах |

### Порівняння вартості:

```
Повний контекст (500K tokens):  ~$1.25 за кожен запит  ×  1000 запитів/день  =  $1,250/день 💸
RAG (2-3K tokens):              ~$0.005 за запит       ×  1000 запитів/день  =  $5/день     ✅
```

> ⚡ **Рішення:** Замість 500K токенів — знайти і передати лише 2-3K **релевантних** токенів. Це дешевше у 250 разів!

## Парадигма RAG (Retrieval-Augmented Generation)

### Базовий принцип:

```
User Query → [Retriever] → Vector DB → Relevant Chunks → [Prompt Augmentation] → LLM → Response
```

**Крок за кроком:**
1. **Indexing (офлайн):** Документи → нарізка на чанки → ембедінги → векторна БД
2. **Retrieval (онлайн):** Запит користувача → ембедінг запиту → пошук найближчих векторів
3. **Generation:** Знайдені чанки + запит → промпт → LLM → відповідь

> 🎯 **Ключова ідея:** Замість того щоб LLM "знала все" — ми **даємо їй правильний контекст** в потрібний момент

## Від Naive RAG до Agentic RAG (Тренд 2026)

| Характеристика | Naive RAG | Agentic RAG |
|---|---|---|
| 🔍 **Пошук** | Одна спроба | Ітеративний (retry + rewrite) |
| 📋 **Планування** | Немає | Агент формує план пошуку |
| ✅ **Валідація** | Немає | Self-reflection на результати |
| 📂 **Джерела** | Одне | Маршрутизація по кількох |
| 🧩 **Складність запитів** | Прості | Multi-hop reasoning |

### Еволюція RAG:

```
Naive RAG        →    Advanced RAG       →    Agentic RAG
(1 пошук,             (Hybrid Search,         (Агент вирішує,
 1 відповідь)          Reranking)              коли/що/як шукати)
```

> 🔑 **Agentic RAG** — агент аналізує запит, формує план пошуку, перевіряє знайдені дані, і у разі потреби — перефразовує та шукає повторно.

## Екосистеми 2026: LlamaIndex vs LangChain/LangGraph

### Розподіл обов'язків:

| Компонент | LlamaIndex 0.14.x | LangChain 1.2.x / LangGraph |
|-----------|-------------------|------------------------------|
| **Data Ingestion** | ✅ SimpleDirectoryReader, Connectors | ✅ Document Loaders |
| **Chunking** | ✅ Node Parsers (Semantic) | ✅ Text Splitters |
| **Indexing** | ✅ VectorStore, PropertyGraph | ❌ (використовує зовнішні) |
| **GraphRAG** | ✅ PropertyGraphIndex | ❌ |
| **Agents** | ✅ FunctionAgent, AgentWorkflow | ✅ create_agent, LangGraph |
| **Orchestration** | ⚠️ Базова | ✅ State Management, Routing |

### Ключовий патерн інтеграції:

```
LlamaIndex Index → Query Engine → LangChain Tool → LangGraph Agent
     (DATA)            (SEARCH)       (BRIDGE)        (ORCHESTRATION)
```

> 🤝 **Разом сильніші:** LlamaIndex для роботи з даними, LangChain/LangGraph для агентної оркестрації

---
# 🔧 Ingestion та Індексування даних

## Етап 1 — Ingestion (Збір даних)

### Готові Document Loaders:

| Формат | LlamaIndex | LangChain |
|--------|-----------|-----------|
| **PDF** | `SimpleDirectoryReader` (вбудований) | `PyPDFLoader`, `PDFMinerLoader` |
| **DOCX** | `SimpleDirectoryReader` (вбудований) | `Docx2txtLoader`, `UnstructuredWordDocumentLoader` |
| **TXT / Markdown** | `SimpleDirectoryReader` | `TextLoader`, `UnstructuredMarkdownLoader` |
| **HTML** | `SimpleDirectoryReader` | `BSHTMLLoader`, `UnstructuredHTMLLoader` |
| **CSV / JSON** | `SimpleDirectoryReader` | `CSVLoader`, `JSONLoader` |
| **SQL** | `DatabaseReader` | `SQLDatabaseLoader` |
| **Notion** | `NotionPageReader` | `NotionDBLoader` |
| **Google Drive** | `GoogleDriveReader` | `GoogleDriveLoader` |
| **Confluence** | `ConfluenceReader` | `ConfluenceLoader` |
| **Slack** | `SlackReader` | `SlackDirectoryLoader` |

> 💡 Будь-який текст можна перетворити на ембедінги та зберегти у векторній БД. Loaders просто спрощують парсинг популярних форматів.

> ⚠️ **Важливо:** Зберігайте метадані (дата, автор, категорія) — вони стануть корисними для фільтрації пізніше!

In [3]:
import os

pdf_path = "./data/retrieval-augmented-generation.pdf"
assert os.path.exists(pdf_path), f"PDF not found at {pdf_path}"

size_kb = os.path.getsize(pdf_path) / 1024
print(f"📄 {pdf_path} ({size_kb:.0f} KB)")
print("✅ Data ready for indexing!")

📄 ./data/retrieval-augmented-generation.pdf (222 KB)
✅ Data ready for indexing!


In [4]:
from llama_index.core import SimpleDirectoryReader

# SimpleDirectoryReader auto-detects file types (PDF, TXT, DOCX, etc.)
documents = SimpleDirectoryReader(
    input_dir="./data",
    recursive=True,
    filename_as_id=True
).load_data()

print(f"📄 Loaded {len(documents)} document fragments")
print(f"\nFirst doc metadata: {documents[0].metadata}")
print(f"First 200 chars: {documents[0].text[:200]}...")

📄 Loaded 6 document fragments

First doc metadata: {'page_label': '1', 'file_name': 'retrieval-augmented-generation.pdf', 'file_path': '/Users/vlad.shanin/Repositories/MULTI-AGENT-SYSTEMS/lesson-5/data/retrieval-augmented-generation.pdf', 'file_type': 'application/pdf', 'file_size': 227758, 'creation_date': '2026-03-14', 'last_modified_date': '2026-03-14'}
First 200 chars: Retrieval-augmented generation
Retrieval-augmented generation (RAG) is a technique that enables large language models (LLMs) to
retrieve and incorporate new information from external data sources.[1] ...


In [5]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("./data/retrieval-augmented-generation.pdf")
langchain_docs = loader.load()

print(f"📄 Loaded {len(langchain_docs)} pages via LangChain PyPDFLoader")
for doc in langchain_docs:
    print(f"  Page: {doc.metadata['page']} | Length: {len(doc.page_content)} chars")

📄 Loaded 6 pages via LangChain PyPDFLoader
  Page: 0 | Length: 3018 chars
  Page: 1 | Length: 2394 chars
  Page: 2 | Length: 2381 chars
  Page: 3 | Length: 1889 chars
  Page: 4 | Length: 3539 chars
  Page: 5 | Length: 2863 chars


## Етап 2 — Chunking (Нарізка тексту)

### Стратегії chunking:

| Стратегія | Як працює | Коли використовувати |
|-----------|-----------|---------------------|
| **Fixed-size** | Ріже кожні N символів, не звертаючи уваги на зміст. Може обрізати речення посередині | Прототипи, прості тексти |
| **Recursive** | Намагається розділити спочатку по `\n\n`, потім `\n`, потім `. `, потім ` ` — зберігає межі абзаців та речень | **Дефолтний вибір** для більшості RAG-систем |
| **Semantic** | Використовує **ембедінги** для вимірювання схожості між послідовними реченнями. Ріже там, де тема змінюється (cosine similarity падає) | Knowledge bases, техдоки з чіткими темами |

### Візуалізація різниці:

```
Вхідний текст: "Auth uses OAuth 2.0. Tokens expire in 1h. | API Gateway routes requests. Rate limit: 1000 RPM."

Fixed-size (200 chars):   "Auth uses OAuth 2.0. Tokens expire in 1h. API Ga" | "teway routes requests..."
                           ❌ обрізано посередині слова!

Recursive:                "Auth uses OAuth 2.0. Tokens expire in 1h." | "API Gateway routes requests. Rate limit: 1000 RPM."
                           ✅ розділено по абзацу/реченню

Semantic:                 [Auth + Tokens → одна тема] | [API Gateway + Rate limit → інша тема]
                           ✅ групує за змістом (потребує embedding API calls)
```

### Як обрати розмір чанка?

```
Занадто великі чанки → шум, нерелевантний контекст
Занадто малі чанки  → втрата контексту, обрізані думки
Goldilocks zone     → 200-1000 tokens залежно від домену
```

> 💡 **Порада:** Починайте з `RecursiveCharacterTextSplitter` — це найкращий баланс простоти та якості. Переходьте на Semantic лише якщо бачите проблеми з якістю retrieval.

In [6]:
# LangChain: comparing 2 chunking strategies
from langchain_text_splitters import RecursiveCharacterTextSplitter

text = """
The authentication system uses OAuth 2.0 protocol for user authorization.
Access tokens expire after 1 hour. Refresh tokens are valid for 30 days.

The API Gateway handles all incoming requests and routes them to appropriate microservices.
Rate limit: 1000 RPM for free tier, 10000 RPM for Enterprise.

PostgreSQL stores the core user data.
Redis is used as a cache for frequently requested data.
"""

recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=30,
    separators=["\n\n", "\n", ". ", " ", ""]
)
recursive_chunks = recursive_splitter.create_documents([text])
print(f"📊 Recursive splitter: {len(recursive_chunks)} chunks\n")

for i, chunk in enumerate(recursive_chunks):
    print(f"--- Chunk {i} ({len(chunk.page_content)} chars) ---")
    print(chunk.page_content.strip())
    print()


📊 Recursive splitter: 3 chunks

--- Chunk 0 (146 chars) ---
The authentication system uses OAuth 2.0 protocol for user authorization.
Access tokens expire after 1 hour. Refresh tokens are valid for 30 days.

--- Chunk 1 (153 chars) ---
The API Gateway handles all incoming requests and routes them to appropriate microservices.
Rate limit: 1000 RPM for free tier, 10000 RPM for Enterprise.

--- Chunk 2 (93 chars) ---
PostgreSQL stores the core user data.
Redis is used as a cache for frequently requested data.



In [7]:
# Semantic Chunking — groups semantically similar sentences together
# This uses embeddings to decide where to split
import importlib, httpx
importlib.reload(httpx)  # reset httpx state that may be polluted by prior cells

from langchain_experimental.text_splitter import SemanticChunker
from langchain_openai import OpenAIEmbeddings

semantic_splitter = SemanticChunker(
    OpenAIEmbeddings(model="text-embedding-3-small"),
    breakpoint_threshold_type="percentile"
)

semantic_chunks = semantic_splitter.create_documents([text])
print(f"📊 Semantic splitter: {len(semantic_chunks)} chunks\n")

for i, chunk in enumerate(semantic_chunks):
    print(f"--- Semantic Chunk {i} ({len(chunk.page_content)} chars) ---")
    print(chunk.page_content.strip())
    print()

print("\n💡 Notice: Semantic chunking groups related sentences together!")

📊 Semantic splitter: 2 chunks

--- Semantic Chunk 0 (395 chars) ---
The authentication system uses OAuth 2.0 protocol for user authorization. Access tokens expire after 1 hour. Refresh tokens are valid for 30 days. The API Gateway handles all incoming requests and routes them to appropriate microservices. Rate limit: 1000 RPM for free tier, 10000 RPM for Enterprise. PostgreSQL stores the core user data. Redis is used as a cache for frequently requested data.

--- Semantic Chunk 1 (0 chars) ---



💡 Notice: Semantic chunking groups related sentences together!


## Етап 3 — Векторні бази даних та Ембедінги

### Що таке ембедінг?

**Embedding** — перетворення тексту у числовий вектор у багатовимірному просторі.

```
"Python is a language"   → [0.12, -0.34, 0.78, ..., 0.45]   (1536 dims)
"JavaScript for web"     → [0.15, -0.31, 0.72, ..., 0.41]   (близький вектор!)
"Cat on the couch"       → [0.89, 0.12, -0.56, ..., -0.23]  (далекий вектор)
```

### Cosine Similarity:
- **1.0** = ідентичні за змістом
- **0.7-0.9** = семантично схожі
- **< 0.3** = не пов'язані

### Популярні Vector DBs:

| База | Тип | Найкраще для |
|------|-----|-------------|
| **FAISS** | In-memory | Швидкі прототипи |
| **Qdrant** | Server (Rust) | Production, фільтрація |
| **Milvus** | Distributed | Масштабні системи |
| **pgvector** | PostgreSQL ext. | Якщо вже є Postgres |
| **Chroma** | Embedded | Простота використання |

In [8]:
from langchain_openai import OpenAIEmbeddings
import numpy as np

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# Create embeddings for 3 sentences
texts = [
    "Python is a programming language",
    "JavaScript is used for web development",
    "A cat is sleeping on the couch"
]

vectors = embeddings.embed_documents(texts)
query_vector = embeddings.embed_query("Which programming language is best?")

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print(f"📐 Vector dimensions: {len(vectors[0])}\n")
print(f"Query: 'Which programming language is best?'\n")

for i, text in enumerate(texts):
    sim = cosine_similarity(query_vector, vectors[i])
    bar = "█" * int(sim * 40)
    print(f"  Similarity with '{text}': {sim:.4f}  {bar}")

print("\n💡 Programming-related texts are much closer to our query!")


📐 Vector dimensions: 1536

Query: 'Which programming language is best?'

  Similarity with 'Python is a programming language': 0.4736  ██████████████████
  Similarity with 'JavaScript is used for web development': 0.3992  ███████████████
  Similarity with 'A cat is sleeping on the couch': 0.0626  ██

💡 Programming-related texts are much closer to our query!


## 💻 Створення індексу — LlamaIndex

Створюємо пошуковий рушій (Query Engine) — це фундамент, який пізніше передамо агентам.

In [9]:
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex, Settings
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai import OpenAI

Settings.llm = OpenAI(model="gpt-5.2", temperature=0)
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-large")

print("📄 Loading documents...")
documents = SimpleDirectoryReader("./data").load_data()
print(f"   Loaded {len(documents)} document fragments.")

print("\n🔨 Building Vector Index...")
index = VectorStoreIndex.from_documents(documents)

llama_query_engine = index.as_query_engine(
    similarity_top_k=3,
    response_mode="compact"  # compact | refine | tree_summarize
)

print("\n🔍 Testing query: 'What is retrieval-augmented generation?'")
test_response = llama_query_engine.query("What is retrieval-augmented generation?")
print(f"\n📝 Response: {test_response}")
print(f"\n📚 Sources ({len(test_response.source_nodes)} nodes):")
for node in test_response.source_nodes:
    print(f"   - Score: {node.score:.4f} | {node.node.metadata.get('file_name', 'unknown')}")

📄 Loading documents...
   Loaded 6 document fragments.

🔨 Building Vector Index...

🔍 Testing query: 'What is retrieval-augmented generation?'

📝 Response: Retrieval-augmented generation (RAG) is a technique for large language models that combines information retrieval with text generation. Instead of relying only on what the model learned during training, it first retrieves relevant information from external sources (such as databases, uploaded documents, or other document collections) and then uses that retrieved content—along with the user’s query—to produce a response. This helps the model use domain-specific or newly updated information and can reduce hallucinations while also enabling more transparent, source-backed answers.

📚 Sources (3 nodes):
   - Score: 0.6547 | retrieval-augmented-generation.pdf
   - Score: 0.6460 | retrieval-augmented-generation.pdf
   - Score: 0.6429 | retrieval-augmented-generation.pdf


## 💻 Створення індексу — LangChain + FAISS

Той самий процес через LangChain — для порівняння підходів.

In [10]:
# LangChain: Build FAISS Index + Retrieval Chain
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

loader = PyPDFLoader("./data/retrieval-augmented-generation.pdf")
documents = loader.load()
print(f"📄 Loaded {len(documents)} pages")

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
chunks = splitter.split_documents(documents)
print(f"✂️  Created {len(chunks)} chunks")

lc_embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
vectorstore = FAISS.from_documents(chunks, lc_embeddings)
print(f"🗃️  FAISS index built!")

vectorstore.save_local("./faiss_index")
print("💾 Index saved to ./faiss_index/")

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
llm = ChatOpenAI(model="gpt-5.2", temperature=0)

prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer the question based on the provided context.\n\nContext:\n{context}"),
    ("human", "{input}"),
])
question_answer_chain = create_stuff_documents_chain(llm, prompt)
qa_chain = create_retrieval_chain(retriever, question_answer_chain)

# 6. Test
print("\n🔍 Testing query: 'What is retrieval-augmented generation?'")
result = qa_chain.invoke({"input": "What is retrieval-augmented generation?"})
print(f"\n📝 Answer: {result['answer']}")
print(f"\n📚 Sources:")
for doc in result['context']:
    print(f"   - Page {doc.metadata.get('page', '?')}")

📄 Loaded 6 pages
✂️  Created 44 chunks


2026-03-15 10:39:41,844 - INFO - Loading faiss.
2026-03-15 10:39:41,872 - INFO - Successfully loaded faiss.


🗃️  FAISS index built!
💾 Index saved to ./faiss_index/

🔍 Testing query: 'What is retrieval-augmented generation?'

📝 Answer: Retrieval-augmented generation (RAG) is a technique where a large language model first retrieves relevant information from a specified set of external documents or data sources and then uses that retrieved material to generate its answer. The retrieved documents supplement what the model already knows from its training data, helping it use more domain-specific and/or up-to-date information (though the system can still misinterpret what it retrieves).

📚 Sources:
   - Page 0
   - Page 5
   - Page 4


---
# 🔬 Просунутий Retrieval та GraphRAG

## Гібридний пошук (Hybrid Search)

### Чому одного типу пошуку недостатньо?

| Тип пошуку | Переваги | Недоліки |
|-----------|----------|---------|
| **Vector (Semantic)** | Розуміє синоніми, контекст | Погано з точними кодами/ID |
| **BM25 (Lexical)** | Точні збіги артикулів, імен | Не розуміє семантику |
| **Hybrid (обидва)** | Найкращий recall | Потрібен reranking |

### Приклад:

```
Запит: "ERROR-4532 authentication failure"

BM25 знайде:  "ERROR-4532" — точний збіг коду помилки ✅
Vector знайде: "authentication problem" — семантична схожість ✅
Hybrid:        обидва результати об'єднані! ✅✅
```

> 🏆 **Hybrid Search — стандарт production RAG у 2026 році**

In [11]:
# LangChain: Hybrid Search with BM25 + Vector (EnsembleRetriever)
from langchain_classic.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever

# Retriever 1: BM25 (lexical — exact keyword matches)
bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 5

# Retriever 2: Vector (semantic — understands context and synonyms)
vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

# Ensemble: combine with weights
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=[0.4, 0.6]  # 40% BM25 + 60% Vector
)

# Test: query with both exact term AND semantic meaning
query = "ERROR-4532 authentication failure"
print(f"🔍 Hybrid search: '{query}'\n")

results = ensemble_retriever.invoke(query)
for i, doc in enumerate(results[:3]):
    print(f"Result {i+1}:")
    print(f"  Source: {doc.metadata.get('source', 'unknown')}")
    print(f"  Content: {doc.page_content[:150]}...")
    print()


🔍 Hybrid search: 'ERROR-4532 authentication failure'

Result 1:
  Source: ./data/retrieval-augmented-generation.pdf
  Content: org/10.1162%2Ftacl_a_00369). Retrieved 15 March 2025.
RAG poisoning
References...

Result 2:
  Source: ./data/retrieval-augmented-generation.pdf
  Content: reliable response. Without specific training, models may generate answers even when they should
indicate uncertainty. According to IBM, this issue can...

Result 3:
  Source: ./data/retrieval-augmented-generation.pdf
  Content: systems may misinterpret the data they retrieve.[2]
1. "What is retrieval-augmented generation?" (https://research.ibm.com/blog/retrieval-augment
ed-g...



## Етап 4 — Reranking (Перевпорядкування)

### Навіщо потрібен Reranking?

```
Пошук (k=10):  [Doc1, Doc2, Doc3, Doc4, Doc5, Doc6, Doc7, Doc8, Doc9, Doc10]
                  ✅     ❌    ✅     ❌    ❌    ✅    ❌    ❌    ❌     ❌

Reranker:       [Doc1, Doc3, Doc6]  ← тільки релевантні, у правильному порядку
```

### Як працює Cross-Encoder Reranker?

1. **Bi-Encoder (embedding):** Кодує query та document окремо → швидко, але неточно
2. **Cross-Encoder:** Кодує `[query + document]` РАЗОМ → повільно, але дуже точно

> 💡 **Стратегія:** Bi-Encoder для пошуку 10 кандидатів → Cross-Encoder для вибору 3 найкращих

In [12]:
# LangChain: Reranking with CrossEncoder
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

# 1. Base retriever (fetch extra candidates — 10 docs)
base_retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

# 2. Cross-Encoder model for reranking
print("⏳ Loading reranker model (first time may take a minute)...")
reranker_model = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-base")
compressor = CrossEncoderReranker(
    model=reranker_model,
    top_n=3  # keep only 3 most relevant
)

# 3. Compression Retriever = Base Search + Reranker
reranking_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=base_retriever
)

# Test
print("\n🔍 Reranking search: 'What loss function is used to train RAG?'\n")
results = reranking_retriever.invoke("What loss function is used to train RAG?")
for i, doc in enumerate(results):
    print(f"Result {i+1}:")
    print(f"  {doc.page_content[:150]}...")
    print()

print("💡 Only the most relevant documents remain after reranking!")

⏳ Loading reranker model (first time may take a minute)...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


🔍 Reranking search: 'What loss function is used to train RAG?'

Result 1:
  uploaded documents, or web sources.[1] According to Ars Technica, "RAG is a way of improving LLM
performance, in essence by blending the LLM process w...

Result 2:
  the LLM's pre-existing training data.[2] This allows LLMs to use domain-specific and/or updated
information that is not available in the training data...

Result 3:
  Retrieval-augmented generation
Retrieval-augmented generation (RAG) is a technique that enables large language models (LLMs) to
retrieve and incorpora...

💡 Only the most relevant documents remain after reranking!


## Проблема розкиданих знань (Multi-hop Queries)

### Коли однієї відповіді недостатньо:

```
Запит: "Як зміна в API сервісу Auth вплине на базу даних Payment?"

                    ┌─────────────┐
                    │  API Spec   │  ← Документ A: Auth Service API
                    └──────┬──────┘
                           │
                    ┌──────▼──────┐
                    │  DB Schema  │  ← Документ B: Database Architecture
                    └──────┬──────┘
                           │
                    ┌──────▼──────┐
                    │ Integration │  ← Документ C: Integration Guide
                    └──────┬──────┘
                           │
                    ┌──────▼──────┐
                    │   Answer    │  ← Синтезована відповідь з 3 джерел
                    └─────────────┘
```

> 🧩 **Naive RAG** не може вирішити таку задачу — потрібен **GraphRAG** або **Agentic RAG**

## Вступ до GraphRAG (Knowledge Graphs + RAG)

### Ключова ідея:

Замість пошуку лише по тексту — будуємо та шукаємо по **графу знань**:

```
[Auth Service] ──DEPENDS_ON──> [PostgreSQL]
      │                              │
      │──CONNECTS_TO──> [Payment GW] │──STORED_IN──> [us-east-1]
      │                              │
      └──OWNED_BY──> [Security Team] └──BACKED_UP──> [Daily]
```

### Переваги GraphRAG:

| Vector Search | GraphRAG |
|---------------|----------|
| "Знайди схожий текст" | "Знайди пов'язані сутності" |
| Не бачить зв'язків | Розуміє відносини |
| Одиночні документи | Multi-hop traversal |

In [13]:
# LlamaIndex: building a Property Graph Index
from llama_index.core import SimpleDirectoryReader, PropertyGraphIndex, Settings
from llama_index.core.indices.property_graph import SchemaLLMPathExtractor
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding
from typing import Literal

Settings.llm = OpenAI(model="gpt-5.2", temperature=0)
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")

documents = SimpleDirectoryReader("./data").load_data()

entities = Literal["MODEL", "DATASET", "METHOD", "METRIC", "COMPONENT"]
relations = Literal["USES", "EVALUATES_ON", "OUTPERFORMS", "PART_OF", "BASED_ON"]

kg_extractor = SchemaLLMPathExtractor(
    llm=Settings.llm,
    possible_entities=entities,
    possible_relations=relations,
    strict=False
)

print("🔨 Building Property Graph Index (this calls LLM to extract entities)...")
graph_index = PropertyGraphIndex.from_documents(
    documents,
    kg_extractors=[kg_extractor],
    show_progress=True
)

graph_query_engine = graph_index.as_query_engine(include_text=True)

print("\n🔍 Query: 'What datasets were used to evaluate RAG?'")
response = graph_query_engine.query(
    "What datasets were used to evaluate RAG?"
)
print(f"\n📝 Response: {response}")

🔨 Building Property Graph Index (this calls LLM to extract entities)...


Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]

Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]

Generating embeddings: 100%|███████████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:00<00:00,  2.21it/s]



🔍 Query: 'What datasets were used to evaluate RAG?'

📝 Response: RAG systems are commonly evaluated using benchmarks such as:

- **BEIR** (a suite of information-retrieval tasks across diverse domains)
- **Natural Questions**
- **Google QA** (for open-domain question answering)


## Комбіновані стратегії пошуку по графу

Один граф можна запитувати різними способами. Нижче комбінуємо дві стратегії:

- **VectorContextRetriever** — знаходить стартову вершину через semantic similarity, потім обходить ребра графу (traversal)
- **LLMSynonymRetriever** — LLM генерує ключові слова та синоніми з запиту, потім шукає відповідні вершини за текстом

In [18]:
# LlamaIndex: combined retrieval strategies over the graph
from llama_index.core.indices.property_graph import (
    LLMSynonymRetriever,
    VectorContextRetriever,
)

# Strategy 1: Vector -> find starting node -> traverse edges
vector_graph_retriever = VectorContextRetriever(
    graph_index.property_graph_store,
    embed_model=Settings.embed_model,
    include_text=True,
    similarity_top_k=2,
    path_depth=2  # traverse 2 hops along graph edges
)

# Strategy 2: LLM analyzes query -> generates keywords -> searches nodes
synonym_retriever = LLMSynonymRetriever(
    graph_index.property_graph_store,
    llm=Settings.llm,
    include_text=True,
    synonym_num=5,
    path_depth=2
)

# Use both strategies together
combined_graph_engine = graph_index.as_query_engine(
    sub_retrievers=[vector_graph_retriever, synonym_retriever],
)

# Multi-hop query
print("🔍 Multi-hop query: 'How does the retriever component affect generation quality?'\n")
response = combined_graph_engine.query(
    "How does the retriever component affect generation quality and what metrics prove it?"
)
print(f"📝 Response: {response}")
print("\n💡 Graph traversal: Retriever -> edges -> Generation -> Metrics")

🔍 Multi-hop query: 'How does the retriever component affect generation quality?'

📝 Response: The retriever component affects generation quality by changing *what evidence the model has available at generation time*. When the retriever returns passages that are more relevant (and less noisy), retrieval-augmented generation systems—such as REPLUG and In-Context Retrieval-Augmented Language Models—can generate answers that are better grounded in the provided material, typically improving overall output quality compared to using no retrieval or weak retrieval.

**What metrics prove it**
The evidence for retrieval impacting generation quality is typically demonstrated through **information retrieval evaluation metrics** applied to the retriever (i.e., how well it finds relevant passages), and through **track-style evaluations** designed for retrieval-augmented generation systems:
- **Information retrieval metrics** (as used in standard IR evaluations) quantify whether the retriever is actu

---
# 🤖 Agentic RAG — Інтеграція пошуку в Агентів

## RAG як інструмент (Tool) агента

### Ключовий патерн:

```
Index → Query Engine → Tool → Agent
```

**Агенту потрібно вказати:**
1. **Коли** використовувати інструмент (через description)
2. **Що** передавати на вхід (query format)
3. **Як** інтерпретувати результат

> 💡 **Tool Description — це промпт для агента!** Чим точніший опис — тим правильніший вибір інструменту.

## Маршрутизація (Routing)

### Логіка агента при виборі інструментів:

```
              Запит користувача
                     │
                     ▼
                ┌─────────┐
                │  Agent  │ ← Аналізує запит
                └────┬────┘
                     │
     ┌───────────────┼────────────────┐
     ▼               ▼                ▼
 [RAG Tool]     [Calculator]   [LLM Memory]
 Корпоративні   Математика     Загальні
 документи                     знання
```

In [ ]:
# LlamaIndex: RouterQueryEngine — automatic query routing
from llama_index.core.query_engine import RouterQueryEngine
from llama_index.core.selectors import LLMSingleSelector
from llama_index.core.tools import QueryEngineTool

# Two genuinely different engines:
# 1. Vector search — fast semantic similarity over chunks
vector_engine = index.as_query_engine(
    similarity_top_k=3,
    response_mode="compact"
)

# 2. Graph search — traverses entity relationships (built earlier)
graph_engine = graph_index.as_query_engine(include_text=True)

# Wrap each into a Tool with clear descriptions
query_engine_tools = [
    QueryEngineTool.from_defaults(
        query_engine=vector_engine,
        description="Best for direct factual questions: definitions, specific details, quotes from the paper."
    ),
    QueryEngineTool.from_defaults(
        query_engine=graph_engine,
        description="Best for relationship questions: how components connect, what depends on what, multi-hop reasoning."
    ),
]

# Router automatically selects the right engine based on query content
router_engine = RouterQueryEngine(
    selector=LLMSingleSelector.from_defaults(),
    query_engine_tools=query_engine_tools,
)

# Test routing — these should go to different engines
queries = [
    "What is retrieval-augmented generation?",          # → vector (factual)
    "How does the retriever component affect the generator's output?",  # → graph (relationship)
]

for q in queries:
    print(f"\n🔍 Query: '{q}'")
    response = router_engine.query(q)
    print(f"📝 Answer: {str(response)[:300]}...")
    selector_result = response.metadata.get('selector_result', None)
    if selector_result:
        print(f"🔀 Router selected: {selector_result}")

## Self-Reflection (Саморефлексія)

### Що робити, коли RAG повертає "сміття"?

```
Naive RAG:
  Query: "deployment process"
  RAG returns: [irrelevant docs about deployment teams]
  LLM: generates hallucinated answer 😱

Agentic RAG with Self-Reflection:
  Query: "deployment process"
  RAG returns: [irrelevant docs]
  Agent thinks: "These docs don't answer the question" 🤔
  Agent rewrites: "step-by-step deployment procedure CI/CD pipeline"
  RAG returns: [relevant docs about CI/CD] ✅
  LLM: generates accurate answer 🎉
```

### Патерн Query Rewriting:
1. Виконати початковий пошук
2. Оцінити релевантність результатів (Self-Reflection)
3. Якщо нерелевантно → перефразувати запит
4. Повторити пошук (до 3 спроб)

## 💻 Об'єднання LlamaIndex та LangChain

**Ключовий момент лекції!** Перетворюємо індекс LlamaIndex у Tool для LangChain агента.

In [20]:
# Bridge: LlamaIndex Index -> LangChain Agent Tool
from llama_index.core.tools import QueryEngineTool, ToolMetadata
from langchain_core.tools import Tool
from langchain_openai import ChatOpenAI
from langchain_classic.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate

# 1. Wrap LlamaIndex Query Engine into a Tool
llama_rag_tool = QueryEngineTool(
    query_engine=llama_query_engine,
    metadata=ToolMetadata(
        name="rag_paper_search",
        description=(
            "Useful for searching the RAG (Retrieval-Augmented Generation) paper. "
            "Input: a specific search query about RAG architecture, evaluation, or methods."
        )
    )
)


# 2. Create a bridge function for LangChain compatibility
def search_rag_paper(query: str) -> str:
    """Execute search query against the LlamaIndex vector store."""
    response = llama_rag_tool.call(query)
    return str(response)


# 3. Define LangChain Tool
langchain_rag_tool = Tool(
    name="RAGPaperSearch",
    func=search_rag_paper,
    description="Searches the RAG paper. Input: a clear search query."
)

tools = [langchain_rag_tool]

# 4. Set up LangChain Agent
llm = ChatOpenAI(model="gpt-5.2", temperature=0)

prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a research assistant.
Always search the paper (RAGPaperSearch) before answering.
Cite sources in your responses."""),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])

agent = create_tool_calling_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

# 5. Run — agent autonomously decides when to use the Tool
print("\n--- Running LangChain Agent with LlamaIndex RAG ---")
response = agent_executor.invoke({
    "input": "What are the two variants of RAG described in the paper and how do they differ?"
})
print(f"\n✅ Final answer: {response['output']}")


--- Running LangChain Agent with LlamaIndex RAG ---


> Entering new AgentExecutor chain...

Invoking: `RAGPaperSearch` with `two variants of RAG described in the paper`


- **In-context RAG**: Mentioned as part of a more reproducible modification of Retro (called **Retro++**), which includes in-context retrieval-augmented generation.

- **Hybrid search RAG**: A variant that combines **vector database (embedding) search** with **traditional text search**, then feeds the combined retrieved text into the language model to reduce missed key facts.
Invoking: `RAGPaperSearch` with `in-context RAG Retro++ more reproducible modification of Retro includes in-context retrieval augmented generation`


Retro has been reported as not reproducible, so it was modified to improve reproducibility. The modified, more reproducible version is called **Retro++**, and it **includes in-context retrieval-augmented generation (in-context RAG)**.
Invoking: `RAGPaperSearch` with `hybrid search RAG combines vec

---
# ⚡ Оптимізація RAG як інструмента для агента

## Векторні бази даних в Production

### Порівняння рішень:

| База | Тип | Переваги | Коли використовувати |
|------|-----|---------|---------------------|
| **FAISS** | In-memory, local | Швидкість, простота | Прототипи, тести |
| **Qdrant** | Server (Rust) | Фільтрація, payload | Production, метадані |
| **Milvus** | Distributed | Горизонтальне масштабування | Big Data, >10M docs |
| **pgvector** | PostgreSQL ext. | Інтеграція з SQL | Якщо вже є Postgres |
| **Pinecone** | Serverless SaaS | Zero ops | Хмарні додатки |

### Критерії вибору:
- **< 100K documents** → FAISS або Chroma
- **100K - 10M documents** → Qdrant або pgvector
- **> 10M documents** → Milvus або Pinecone
- **Потрібна фільтрація** → Qdrant (native metadata filtering)

## Фільтрація шуму та Self-Querying Retrieval

### Проблема "сліпого" пошуку:

```
Query: "Find API requirements from engineering in 2025"

❌ Naive:  Шукає семантично схожий текст — може повернути HR policy від 2023

✅ Self-Querying:
   LLM аналізує запит →
   Генерує фільтри: year=2025 AND department="engineering" AND doc_type="api_spec"
   Потім семантичний пошук ТІЛЬКИ серед відфільтрованих документів!
```

In [21]:
# Self-Querying concept: LLM extracts metadata filters from natural language
# Note: SelfQueryRetriever requires specific vector stores (Chroma, Pinecone, Qdrant)
# With FAISS, we demonstrate the concept manually

from langchain_openai import ChatOpenAI
import json

llm = ChatOpenAI(model="gpt-5.2", temperature=0)

# Step 1: LLM extracts filters from a natural language query
query = "Find information about retrieval from pages 1-3"
filter_prompt = f"""Extract search query and metadata filters from this user question.
Available metadata fields: page (integer), source (string).

User question: {query}

Respond in JSON: {{"query": "semantic search query", "filters": {{"field": "value"}}}}"""

response = llm.invoke(filter_prompt)
print(f"🔍 User query: '{query}'")
print(f"🤖 LLM extracted: {response.content}\n")

# Step 2: Apply filters manually with FAISS
# FAISS supports filtering via search kwargs with a filter function
results = vectorstore.similarity_search(
    query="retrieval",
    k=5,
    filter={"page": 1}  # FAISS supports exact match filtering on metadata
)

print(f"📄 Results filtered to page 1:")
for doc in results[:3]:
    print(f"   Page {doc.metadata.get('page')}: {doc.page_content[:100]}...")

print("\n💡 In production, use Qdrant or Chroma for full SelfQueryRetriever support")
print("   with automatic filter extraction (comparators: >, <, ==, IN, etc.)")

🔍 User query: 'Find information about retrieval from pages 1-3'
🤖 LLM extracted: {
  "query": "information about retrieval",
  "filters": {
    "page": [1, 2, 3]
  }
}

📄 Results filtered to page 1:
   Page 1: memory and self-improvement to learn from previous retrievals.
Finally, the LLM can generate output ...
   Page 1: select the most relevant documents that will be used to
augment the query.[2][3] This comparison can...

💡 In production, use Qdrant or Chroma for full SelfQueryRetriever support
   with automatic filter extraction (comparators: >, <, ==, IN, etc.)


## Стратегії просунутого Chunking для збереження контексту

### Parent Document Retriever:

```
Пошук:   [маленький chunk 400 chars] — точний match
Відповідь: [великий parent 2000 chars] — повний контекст!
```

### Sentence-Window Retrieval:

```
Пошук за реченнями → повернення з контекстом ±5 речень навколо
```

> 🎯 **Ідея:** Шукаємо по дрібних чанках (точність), але повертаємо великі (контекст)

In [22]:
# LangChain: Parent Document Retriever
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.storage import InMemoryStore
from langchain_community.document_loaders import PyPDFLoader

# Two-level splitting:
# Parent chunks — large (2000 chars), provide context to LLM
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=200)
# Child chunks — small (300 chars), provide precision for vector search
child_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)

parent_embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
parent_vectorstore = FAISS.from_texts(["placeholder"], parent_embeddings)
docstore = InMemoryStore()

parent_retriever = ParentDocumentRetriever(
    vectorstore=parent_vectorstore,
    docstore=docstore,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

# Load the PDF
parent_docs = PyPDFLoader("./data/retrieval-augmented-generation.pdf").load()

# Add documents
parent_retriever.add_documents(parent_docs)

# How it works during search:
# 1. Vector search finds small child chunk (300 chars) — precise match
# 2. Retriever returns the entire parent chunk (2000 chars) — full context!
print("🔍 Parent Document Retriever: 'How does RAG handle knowledge-intensive tasks?'\n")
results = parent_retriever.invoke("How does RAG handle knowledge-intensive tasks?")
for doc in results:
    print(f"📄 Parent chunk ({len(doc.page_content)} chars):")
    print(f"   {doc.page_content[:200]}...")
    print()

print("💡 Notice: we searched by small chunks but got back large parent documents!")

🔍 Parent Document Retriever: 'How does RAG handle knowledge-intensive tasks?'

📄 Parent chunk (1977 chars):
   Retrieval-augmented generation
Retrieval-augmented generation (RAG) is a technique that enables large language models (LLMs) to
retrieve and incorporate new information from external data sources.[1] ...

📄 Parent chunk (1975 chars):
   Overview of RAG process, combining
external documents and user input into
an LLM prompt to get tailored output
Retrieval-augmented generation (RAG) enhances large language models (LLMs) by incorporati...

💡 Notice: we searched by small chunks but got back large parent documents!


## 💻 RAG Tool з Reranking

Створюємо надійний `Tool`, який передамо агентам на наступних заняттях.

In [23]:
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_core.tools import Tool

# 1. Base retriever (fetch 10 candidates as buffer)
base_retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

# 2. Reranking model (filters out noise)
model = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-base")
compressor = CrossEncoderReranker(model=model, top_n=3)

# 3. Compression retriever = Base Search + Reranking
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=base_retriever
)


# 4. Wrap into a Tool with a clear description for the agent
def optimized_rag_search(query: str) -> str:
    """Search the RAG paper with automatic reranking."""
    docs = compression_retriever.invoke(query)
    if not docs:
        return "No documents found for this query. Try rephrasing."

    results = []
    for i, doc in enumerate(docs, 1):
        page = doc.metadata.get('page', '?')
        results.append(f"[Source {i}: page {page}]\n{doc.page_content}")

    return "\n\n---\n\n".join(results)


robust_rag_tool = Tool(
    name="Optimized_RAG_Search",
    func=optimized_rag_search,
    description=(
        "Use for precise search across the RAG paper. "
        "Automatically filters out irrelevant noise via reranking. "
        "Input: a specific search query in English or Ukrainian."
    )
)

# Test our production tool
print("🔍 Testing robust_rag_tool: 'RAG-Sequence vs RAG-Token'\n")
result = robust_rag_tool.run("RAG-Sequence vs RAG-Token")
print(result)

print("\n" + "="*60)
print("✅ Production-ready RAG Tool created!")

2026-03-15 10:44:24,217 - WARNING - Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

🔍 Testing robust_rag_tool: 'RAG-Sequence vs RAG-Token'

[Source 1: page 0]
uploaded documents, or web sources.[1] According to Ars Technica, "RAG is a way of improving LLM
performance, in essence by blending the LLM process with a web search or other document look-up
process to help LLMs stick to the facts." This method helps reduce AI hallucinations,[3] which have
caused chatbots to describe policies that don't exist, or recommend nonexistent legal cases to lawyers that
are looking for citations to support their arguments.[4]

---

[Source 2: page 0]
Retrieval-augmented generation
Retrieval-augmented generation (RAG) is a technique that enables large language models (LLMs) to
retrieve and incorporate new information from external data sources.[1] With RAG, LLMs first refer to a
specified set of documents, then respond to user queries. These documents supplement information from
the LLM's pre-existing training data.[2] This allows LLMs to use domain-specific and/or updated

---

[Sourc

## 💻  Повний Agentic RAG з LlamaIndex FunctionAgent

Альтернативний підхід — повністю на LlamaIndex, без LangChain.

In [24]:
from llama_index.core.agent.workflow import FunctionAgent
from llama_index.core.tools import QueryEngineTool, FunctionTool
from llama_index.llms.openai import OpenAI

# 1. RAG Tool (from our index)
rag_tool = QueryEngineTool.from_defaults(
    query_engine=llama_query_engine,
    name="rag_paper_search",
    description="Search the RAG paper. Use for any questions about the paper's content."
)


# 2. Simple calculator as a Tool
def calculate(expression: str) -> str:
    """Evaluate a math expression. Input: string like '2 + 2' or '1024 * 3'."""
    try:
        # In production, use ast.literal_eval or sympy for safety
        result = eval(expression)
        return f"Result: {result}"
    except Exception as e:
        return f"Calculation error: {e}"


calc_tool = FunctionTool.from_defaults(fn=calculate, name="calculator")

# 3. Create Agentic RAG
agent = FunctionAgent(
    tools=[rag_tool, calc_tool],
    llm=OpenAI(model="gpt-5.2"),
    system_prompt="""You are a research assistant analyzing academic papers.
Always search the paper before answering questions.
If you need to compute something, use the calculator.
Provide structured answers with source references."""
)

# 4. Run (async — LlamaIndex FunctionAgent uses async workflows)
import asyncio


async def main():
    response = await agent.run(
        "How many datasets were used to evaluate RAG and what were the results?"
    )
    print(f"📝 Agent response:\n{response}")


# In Colab, use await directly or nest_asyncio
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass

asyncio.run(main())

📝 Agent response:
### How many datasets were used?
The paper section available via search lists **3 evaluation datasets/benchmarks** for RAG:
1. **BEIR**
2. **Natural Questions**
3. **Google QA**  
**Source:** paper search results mentioning “BEIR”, “Natural Questions”, and “Google QA” as evaluation datasets for RAG. (RAG paper search: “evaluate RAG datasets…”)

### What were the results?
In the retrieved paper text, these datasets are **only named as commonly used benchmarks** to evaluate RAG (e.g., retrievability, retrieval accuracy, generative quality), but **no quantitative results/scores are provided** for any of them in the accessible excerpts.  
**Sources:** paper search results for “Natural Questions results”, “BEIR results”, and “Google QA results” explicitly indicate **no specific scores/results are provided**, only that they are used for evaluation.


---
# 📋 Підсумки

## Що ми вивчили сьогодні:

### Еволюція RAG:

| Рівень | Що робить | Ключові технології |
|--------|-----------|-------------------|
| **Naive RAG** | Одна спроба пошуку, лінійний пайплайн | Vector Search |
| **Advanced RAG** | Hybrid Search + Reranking + Parent-Child | BM25, CrossEncoder, Self-Query |
| **Agentic RAG** | Агент сам вирішує коли/що/як шукати | FunctionAgent, create_agent |
| **GraphRAG** | Знає зв'язки між сутностями | PropertyGraphIndex, Multi-hop |

### Головний висновок:

> 🔑 **RAG у 2026 — це динамічний процес (агентна оркестрація + графи), а не просто векторна база даних. Це повноцінна довгострокова пам'ять для MAS.**

### Що далі?

> 🔮 **Наступна лекція:** Інструменти оркестрації AI агентів.
> Ми візьмемо `robust_rag_tool` звідси та побудуємо мультиагентний workflow у **LangGraph**.

---

### Корисні ресурси:
- 📖 [LlamaIndex Documentation](https://docs.llamaindex.ai/)
- 📖 [LangChain Documentation](https://python.langchain.com/)
- 📖 [FAISS Wiki](https://github.com/facebookresearch/faiss/wiki)
